In [2]:
# Cell 1: Mount Drive and set project paths
from google.colab import drive
drive.mount('/content/drive')

PROJECT_PATH = '/content/drive/MyDrive/financial-news-sentiment-stock-predictor'
DATA_PATH = f'{PROJECT_PATH}/data'

import pandas as pd
import requests
from datetime import datetime, timedelta
from google.colab import userdata

# Load API key securely
API_KEY = userdata.get('NEWSAPI_KEY')
print("API key loaded:", API_KEY is not None)

# Date range for news (last 30 days - NewsAPI free tier limit)
from_date = (datetime.now() - timedelta(days=29)).strftime('%Y-%m-%d')
to_date = datetime.now().strftime('%Y-%m-%d')
print(f"Fetching news from {from_date} to {to_date}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
API key loaded: True
Fetching news from 2026-05-17 to 2026-06-15


In [3]:
# Cell 2: Define search queries per ticker
# Multiple queries per ticker increases headline coverage
TICKER_QUERIES = {
    'AAPL': [
        'Apple AAPL stock',
        'Apple Inc earnings',
        'iPhone Apple revenue',
        'Tim Cook Apple'
    ],
    'MSFT': [
        'Microsoft MSFT stock',
        'Microsoft earnings revenue',
        'Azure Microsoft cloud',
        'Satya Nadella Microsoft'
    ],
    'GOOGL': [
        'Google GOOGL stock',
        'Alphabet Google earnings',
        'Google AI search revenue',
        'Sundar Pichai Google'
    ],
    'AMZN': [
        'Amazon AMZN stock',
        'Amazon earnings AWS',
        'Amazon retail revenue',
        'Andy Jassy Amazon'
    ],
    'TSLA': [
        'Tesla TSLA stock',
        'Tesla earnings deliveries',
        'Tesla EV Elon Musk',
        'Tesla revenue production'
    ]
}

# Relevance keywords for filtering per ticker
TICKER_KEYWORDS = {
    'AAPL':  ['apple', 'aapl', 'iphone', 'tim cook', 'mac', 'ipad'],
    'MSFT':  ['microsoft', 'msft', 'azure', 'satya', 'windows', 'copilot'],
    'GOOGL': ['google', 'googl', 'alphabet', 'sundar', 'youtube', 'gemini'],
    'AMZN':  ['amazon', 'amzn', 'aws', 'andy jassy', 'prime', 'alexa'],
    'TSLA':  ['tesla', 'tsla', 'elon', 'musk', 'electric vehicle', 'ev']
}

print("Search queries defined for tickers:")
for ticker, queries in TICKER_QUERIES.items():
    print(f"  {ticker}: {len(queries)} queries")

Search queries defined for tickers:
  AAPL: 4 queries
  MSFT: 4 queries
  GOOGL: 4 queries
  AMZN: 4 queries
  TSLA: 4 queries


In [4]:
# Cell 3: Fetch news headlines for all tickers
import time

def fetch_ticker_news(ticker, queries, api_key, from_date, to_date):
    """Fetch news articles for a ticker using multiple search queries."""
    url = "https://newsapi.org/v2/everything"
    articles = []

    for query in queries:
        params = {
            "q": query,
            "from": from_date,
            "to": to_date,
            "language": "en",
            "sortBy": "publishedAt",
            "pageSize": 100,
            "apiKey": api_key
        }
        response = requests.get(url, params=params)
        result = response.json()

        if result.get("status") == "ok":
            articles.extend(result.get("articles", []))
            print(f"  [{ticker}] '{query}' → "
                  f"{result.get('totalResults', 0)} results, "
                  f"fetched {len(result.get('articles', []))}")
        else:
            print(f"  [{ticker}] '{query}' → ERROR: {result.get('message')}")

        time.sleep(0.5)  # avoid hitting API rate limit

    return articles

# Fetch for all tickers
all_ticker_articles = {}
for ticker in TICKER_QUERIES:
    print(f"\nFetching news for {ticker}...")
    articles = fetch_ticker_news(
        ticker,
        TICKER_QUERIES[ticker],
        API_KEY,
        from_date,
        to_date
    )
    all_ticker_articles[ticker] = articles
    print(f"  Total raw articles for {ticker}: {len(articles)}")


Fetching news for AAPL...
  [AAPL] 'Apple AAPL stock' → 92 results, fetched 90
  [AAPL] 'Apple Inc earnings' → 51 results, fetched 50
  [AAPL] 'iPhone Apple revenue' → 146 results, fetched 96
  [AAPL] 'Tim Cook Apple' → 335 results, fetched 97
  Total raw articles for AAPL: 333

Fetching news for MSFT...
  [MSFT] 'Microsoft MSFT stock' → 139 results, fetched 100
  [MSFT] 'Microsoft earnings revenue' → 339 results, fetched 100
  [MSFT] 'Azure Microsoft cloud' → 548 results, fetched 100
  [MSFT] 'Satya Nadella Microsoft' → 242 results, fetched 100
  Total raw articles for MSFT: 400

Fetching news for GOOGL...
  [GOOGL] 'Google GOOGL stock' → 63 results, fetched 63
  [GOOGL] 'Alphabet Google earnings' → 115 results, fetched 96
  [GOOGL] 'Google AI search revenue' → 430 results, fetched 98
  [GOOGL] 'Sundar Pichai Google' → 185 results, fetched 100
  Total raw articles for GOOGL: 357

Fetching news for AMZN...
  [AMZN] 'Amazon AMZN stock' → 100 results, fetched 100
  [AMZN] 'Amazon earnin

In [5]:
# Cell 4: Clean, filter and deduplicate articles for all tickers
import re

def clean_text(text):
    """Clean headline/description text."""
    if pd.isna(text) or text is None:
        return ""
    text = re.sub(r'<.*?>', '', str(text))      # remove HTML tags
    text = re.sub(r'http\S+', '', text)          # remove URLs
    text = re.sub(r'\s+', ' ', text)             # normalize whitespace
    return text.strip()

def is_relevant(headline, description, keywords):
    """Check if article is relevant to the ticker."""
    text = f"{headline} {description}".lower()
    return any(kw.lower() in text for kw in keywords)

all_ticker_dfs = {}

for ticker, articles in all_ticker_articles.items():
    rows = []
    for article in articles:
        headline = clean_text(article.get('title', ''))
        description = clean_text(article.get('description', ''))
        source = article.get('source', {}).get('name', '')
        date = article.get('publishedAt', '')[:10]

        # Skip empty or removed articles
        if not headline or headline == '[Removed]':
            continue

        # Apply relevance filter
        if not is_relevant(headline, description,
                           TICKER_KEYWORDS[ticker]):
            continue

        rows.append({
            'date': date,
            'ticker': ticker,
            'headline': headline,
            'description': description if description else
                          'No description available',
            'source': source
        })

    # Create DataFrame and deduplicate
    df = pd.DataFrame(rows)
    df = df.drop_duplicates(subset=['headline']).reset_index(drop=True)
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values('date').reset_index(drop=True)

    all_ticker_dfs[ticker] = df
    print(f"{ticker}: {len(articles)} raw → "
          f"{len(df)} after cleaning & deduplication")

print(f"\nTotal articles across all tickers: "
      f"{sum(len(df) for df in all_ticker_dfs.values())}")

AAPL: 333 raw → 164 after cleaning & deduplication
MSFT: 400 raw → 151 after cleaning & deduplication
GOOGL: 357 raw → 137 after cleaning & deduplication
AMZN: 346 raw → 107 after cleaning & deduplication
TSLA: 267 raw → 153 after cleaning & deduplication

Total articles across all tickers: 712


In [7]:
# Cell 5: Combine all ticker DataFrames and analyze daily coverage
news_data = pd.concat(all_ticker_dfs.values(),
                      ignore_index=True)
news_data = news_data.sort_values(
    ['ticker', 'date']).reset_index(drop=True)

print(f"Combined news dataset shape: {news_data.shape}")
print(f"\nArticles per ticker (after cleaning):")
print(news_data['ticker'].value_counts().sort_index())

print(f"\nDate range: {news_data['date'].min().date()} "
      f"to {news_data['date'].max().date()}")

# Define tickers list explicitly in case session restarted
TICKERS = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA']

print(f"\nDaily coverage per ticker (avg headlines/day):")
for ticker in TICKERS:
    subset = news_data[news_data['ticker'] == ticker]
    days = subset['date'].nunique()
    avg = len(subset) / days if days > 0 else 0
    print(f"  {ticker}: {len(subset)} articles across "
          f"{days} days ({avg:.1f} avg/day)")

news_data.head(10)

Combined news dataset shape: (712, 5)

Articles per ticker (after cleaning):
ticker
AAPL     164
AMZN     107
GOOGL    137
MSFT     151
TSLA     153
Name: count, dtype: int64

Date range: 2026-05-17 to 2026-06-14

Daily coverage per ticker (avg headlines/day):
  AAPL: 164 articles across 24 days (6.8 avg/day)
  MSFT: 151 articles across 19 days (7.9 avg/day)
  GOOGL: 137 articles across 26 days (5.3 avg/day)
  AMZN: 107 articles across 24 days (4.5 avg/day)
  TSLA: 153 articles across 28 days (5.5 avg/day)


,date,ticker,headline,description,source
0,2026-05-19,AAPL,Trump Organization’s managed account executes ...,President Trump's financial disclosure for the...,Macdailynews.com
1,2026-05-20,AAPL,Apple Inc. (AAPL): A Top Stock Pick by Graham ...,No description available,Yahoo Entertainment
2,2026-05-20,AAPL,KeyBanc Turns Cautious on Apple Stock as Valua...,"Apple's valuation looks ""stretched,"" according...",Barchart.com
3,2026-05-21,AAPL,Nvidia's $80 billion stock buyback and bigger ...,Nvidia upping its cash payouts could ignite th...,Yahoo Entertainment
4,2026-05-21,AAPL,Is Apple Inc. (AAPL) One of Louis Navellier’s ...,No description available,Yahoo Entertainment
5,2026-05-22,AAPL,"Apple Stock at 52-Week High: Buy, Sell or Hold?","Apple (NASDAQ:AAPL) at $304.99 is a hold, with...",24/7 Wall St.
6,2026-05-22,AAPL,Microsoft vs. Apple Stock: The Numbers Reveal ...,Microsoft vs. Apple stock: Compare fundamental...,Barchart.com
7,2026-05-22,AAPL,Analyzing an Apple Stock Butterfly Spread: Ris...,The long call butterfly spread is a defined-ri...,Barchart.com
8,2026-05-25,AAPL,"Dear Apple Stock Fans, Mark Your Calendars for...",Apple is cutting prices in China ahead of the ...,Barchart.com
9,2026-05-26,AAPL,5 High-Yield Stocks and ETFs to Buy and Hold f...,Five high-yield names spanning pharmaceuticals...,MarketBeat


In [8]:
# Cell 6: Final validation and save news data
# Check null values
print("Null values:")
print(news_data.isnull().sum())

# Check duplicate headlines within same ticker
dupes = news_data.duplicated(subset=['ticker', 'headline']).sum()
print(f"\nDuplicate ticker-headline pairs: {dupes}")

# Articles per day across all tickers
print(f"\nTotal articles per date (all tickers combined):")
daily_counts = news_data.groupby('date').size()
print(f"  Min: {daily_counts.min()} | "
      f"Max: {daily_counts.max()} | "
      f"Mean: {daily_counts.mean():.1f}")

print(f"\nFinal dataset shape: {news_data.shape}")

# Save to CSV
news_data.to_csv(f'{DATA_PATH}/news_data.csv', index=False)
print(f"\nSaved to {DATA_PATH}/news_data.csv")

Null values:
date           0
ticker         0
headline       0
description    0
source         0
dtype: int64

Duplicate ticker-headline pairs: 0

Total articles per date (all tickers combined):
  Min: 3 | Max: 74 | Mean: 24.6

Final dataset shape: (712, 5)

Saved to /content/drive/MyDrive/financial-news-sentiment-stock-predictor/data/news_data.csv
